# End-to-End Real-Dataset Analysis

This notebook verifies the independent CICIDS2017 and BoT-IoT baselines plus one simultaneous unified model trained on both source schemas with a common label taxonomy. The configured row caps are used to remain within local memory.

In [1]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT

PosixPath('/Users/kousiksamanta/Documents/FINAL DISSERTATION')

## Verify Environment

In [2]:
subprocess.run(['make', 'doctor'], cwd=PROJECT_ROOT, check=True)

Python: /Users/kousiksamanta/Documents/FINAL DISSERTATION/.venv/bin/python


Dependencies: OK


CompletedProcess(args=['make', 'doctor'], returncode=0)

## Verify Generated Baselines and Unified Pipeline

The full experiments are run from the project root with `make all`. This notebook verifies the generated artifacts and refreshes the cross-dataset summary without retraining the three model suites every time the notebook is executed.

In [3]:
import pandas as pd

required_artifacts = []
for dataset in ('cicids2017', 'bot-iot', 'combined'):
    required_artifacts.extend([
        PROJECT_ROOT / 'data' / 'processed' / dataset / 'preprocessing_manifest.json',
        PROJECT_ROOT / 'models' / 'saved' / dataset / 'best_model.json',
        PROJECT_ROOT / 'results' / dataset / 'metrics' / 'ids_model_comparison.csv',
        PROJECT_ROOT / 'reports' / dataset / 'results_summary.md',
    ])

artifact_status = pd.DataFrame(
    {
        'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in required_artifacts],
        'exists': [path.exists() for path in required_artifacts],
    }
)
display(artifact_status)
assert artifact_status['exists'].all(), 'Run `make all` from the project root to regenerate missing artifacts.'
subprocess.run(['make', 'summary'], cwd=PROJECT_ROOT, check=True)


,artifact,exists
0,data/processed/cicids2017/preprocessing_manife...,True
1,models/saved/cicids2017/best_model.json,True
2,results/cicids2017/metrics/ids_model_compariso...,True
3,reports/cicids2017/results_summary.md,True
4,data/processed/bot-iot/preprocessing_manifest....,True
5,models/saved/bot-iot/best_model.json,True
6,results/bot-iot/metrics/ids_model_comparison.csv,True
7,reports/bot-iot/results_summary.md,True
8,data/processed/combined/preprocessing_manifest...,True
9,models/saved/combined/best_model.json,True


.venv/bin/python -m soc_ready_ids.evaluation.dataset_summary --config config.yaml


/Users/kousiksamanta/Documents/FINAL DISSERTATION/reports/results_summary.md


CompletedProcess(args=['make', 'summary'], returncode=0)

## Compare Results

In [4]:
from IPython.display import Markdown, display

display(Markdown((PROJECT_ROOT / 'reports' / 'results_summary.md').read_text(encoding='utf-8')))

# Combined Real-Dataset Results

CICIDS2017 and BoT-IoT were evaluated independently, and the combined run trained one unified model on both source schemas with a common label taxonomy. Artifacts are isolated by run.

| dataset    |   retained_rows |   classes | best_model    |   accuracy |   f1_macro |   binary_attack_f1_macro |   alert_reduction_rate |   tp_preservation_rate |   explanation_scs |
|:-----------|----------------:|----------:|:--------------|-----------:|-----------:|-------------------------:|-----------------------:|-----------------------:|------------------:|
| cicids2017 |          100000 |        15 | random_forest |    0.99505 |   0.804665 |                 0.99156  |                     77 |                    100 |             0.89  |
| bot-iot    |          100000 |         4 | random_forest |    1       |   1        |                 1        |                     77 |                    100 |             1     |
| combined   |          100000 |         5 | xgboost_ids   |    0.99875 |   0.984508 |                 0.998934 |                     78 |                    100 |             0.953 |

> Compare results with care: the configured samples preserve each dataset's observed class distribution, and the supplied BoT-IoT files contain very few benign rows. The combined run uses a broad common label taxonomy and equal row allocation from each source.

## cicids2017

- Dataset report: `reports/cicids2017/results_summary.md`
- Processed data: `data/processed/cicids2017/`
- Models: `models/saved/cicids2017/`
- Results: `results/cicids2017/`
- Classes: BENIGN, Bot, DDoS, DoS GoldenEye, DoS Hulk, DoS Slowhttptest, DoS slowloris, FTP-Patator, Heartbleed, Infiltration, PortScan, SSH-Patator, Web Attack � Brute Force, Web Attack � Sql Injection, Web Attack � XSS
- Class distribution: BENIGN=87793, DDoS=7270, DoS Hulk=3650, DoS GoldenEye=232, FTP-Patator=207, PortScan=201, Bot=124, Web Attack � Brute Force=117, SSH-Patator=113, DoS slowloris=106, DoS Slowhttptest=102, Web Attack � XSS=64, Infiltration=10, Web Attack � Sql Injection=6, Heartbleed=5

## bot-iot

- Dataset report: `reports/bot-iot/results_summary.md`
- Processed data: `data/processed/bot-iot/`
- Models: `models/saved/bot-iot/`
- Results: `results/bot-iot/`
- Classes: BENIGN, DDoS, DoS, Reconnaissance
- Class distribution: DDoS=52689, DoS=44607, Reconnaissance=2689, BENIGN=15

## combined

- Dataset report: `reports/combined/results_summary.md`
- Processed data: `data/processed/combined/`
- Models: `models/saved/combined/`
- Results: `results/combined/`
- Classes: BENIGN, DDoS, DoS, Other Attack, Reconnaissance
- Class distribution: BENIGN=43883, DDoS=30003, DoS=24326, Reconnaissance=1428, Other Attack=360
- Source distribution: cicids2017=50000, bot-iot=50000


## Wazuh Validation

In [5]:
for dataset in ('cicids2017', 'bot-iot', 'combined'):
    subprocess.run(['make', 'wazuh', f'DATASET={dataset}'], cwd=PROJECT_ROOT, check=True)

.venv/bin/python wazuh/validate_integration.py --project-root . --dataset cicids2017


{
  "validated_xml": [
    "custom_decoder.xml",
    "custom_rules.xml",
    "ossec.conf"
  ],
  "mock_event": {
    "integration": "soc-ready-ids",
    "wazuh_alert_id": "mock-wazuh-alert-001",
    "wazuh_rule_id": "100103",
    "agent_id": "000",
    "agent_name": "wazuh-manager",
    "timestamp": "2026-06-08T12:00:00+00:00",
    "alert_id": "mock-wazuh-alert-001",
    "src_ip": "10.0.2.208",
    "dst_ip": "192.168.10.22",
    "dst_port": "8080",
    "attack_type": "BENIGN",
    "confidence": 0.917813,
    "risk_score": 62.71,
    "risk_tier": "Medium",
    "cluster_id": -1,
    "is_suppressed": false,
    "triage_duration_ms": 3745.134,
    "explanation_text": "This alert was classified as BENIGN with 91.8% confidence because Init Fwd Win Bytes is at the 44th percentile and pushed risk up, Init Bwd Win Bytes is at the 61th percentile and pushed risk up, Fwd IAT Max is at the 20th percentile and pushed risk up. Benign traffic is expected network activity with no attack signature in t

.venv/bin/python wazuh/validate_integration.py --project-root . --dataset bot-iot


{
  "validated_xml": [
    "custom_decoder.xml",
    "custom_rules.xml",
    "ossec.conf"
  ],
  "mock_event": {
    "integration": "soc-ready-ids",
    "wazuh_alert_id": "mock-wazuh-alert-001",
    "wazuh_rule_id": "100103",
    "agent_id": "000",
    "agent_name": "wazuh-manager",
    "timestamp": "2026-06-08T12:00:00+00:00",
    "alert_id": "mock-wazuh-alert-001",
    "src_ip": "10.0.2.208",
    "dst_ip": "192.168.10.22",
    "dst_port": "8080",
    "attack_type": "DDoS",
    "confidence": 0.51,
    "risk_score": 62.9,
    "risk_tier": "Medium",
    "cluster_id": -1,
    "is_suppressed": false,
    "triage_duration_ms": 1357.045,
    "explanation_text": "This alert was classified as DDoS with 51.0% confidence because seq is at the 0th percentile and pushed risk down, dpkts is at the 84th percentile and pushed risk up, pkts is at the 0th percentile and pushed risk up. DDoS activity attempts to exhaust a service using high-volume distributed traffic. Recommended first response: Check 

.venv/bin/python wazuh/validate_integration.py --project-root . --dataset combined


{
  "validated_xml": [
    "custom_decoder.xml",
    "custom_rules.xml",
    "ossec.conf"
  ],
  "mock_event": {
    "integration": "soc-ready-ids",
    "wazuh_alert_id": "mock-wazuh-alert-001",
    "wazuh_rule_id": "100103",
    "agent_id": "000",
    "agent_name": "wazuh-manager",
    "timestamp": "2026-06-08T12:00:00+00:00",
    "alert_id": "mock-wazuh-alert-001",
    "src_ip": "10.0.2.208",
    "dst_ip": "192.168.10.22",
    "dst_port": "8080",
    "attack_type": "BENIGN",
    "confidence": 0.999038,
    "risk_score": 65.96,
    "risk_tier": "High",
    "cluster_id": -1,
    "is_suppressed": false,
    "triage_duration_ms": 1314.642,
    "explanation_text": "This alert was classified as BENIGN with 99.9% confidence because proto=missing is at the 100th percentile and pushed risk up, Init Fwd Win Bytes is at the 23th percentile and pushed risk up, flgs=missing is at the 100th percentile and pushed risk up. Benign traffic is expected network activity with no attack signature in this 